# Q2 충돌 확인 + 같은 호출 수 재질문 — dev40

**준비/CPU 검증 완료. 새 구조의 실제 모델 효과는 아직 미검증입니다.**

실행 파일: `lvlm_q2_conflict_evidence_dev_colab.ipynb`

L4/A100 GPU 선택 → 모두 실행 → Drive 연결 허용. **이 노트북은 실제 모델 추론을 실행합니다.**
기존 dev40 답을 재사용하고, 첫 판단과 정의 재검토가 다른 10건에서만 새 40회 호출합니다.
결과 파일: `q2_conflict_evidence_dev40_results.zip`. 새 GitHub 토큰/수동 라벨 입력은 필요 없습니다.

- 후보별 사진 지지 확인2회 vs 순차 재질문2회. 두 정책은 동일 사례·모델·이미지·최대출력80토큰·호출 수입니다. 질문 내용/길이·실제 시간은 다릅니다.
- 한 후보만 지지될 때 그 답을 사용. 둘 다 지지/둘 다 미지지는 초기 답 유지. 무효/오류는 숨기지 않고 정책 제외 및 원시 ZIP 보존.
- 초기/정의 재검토/기존 cue 답도 비교합니다. 초기 조건보다만 좋아져도 성공이라고 하지 않습니다.
- 과거 240장의 77충돌: 정답→오답18, 오답→정답42, 다른 오답17. 이미 분석한 dev이며 최종 holdout이 아닙니다.
- Q1 신규 호출0·원래 답 고정. Q3/gold/학습0. 과거 두 판단의 모델 revision은 미기록이며 새 비교 호출만 같은 snapshot으로 고정합니다.
- 끊기면 같은 설정으로 다시 실행. 저장된 성공/무효 호출은 재사용하고 오류만 재시도합니다. 추가200장은 이40장 결과 검토 후 따로 준비합니다.
- 원본 ZIP 두 개는 Drive에 있어야 합니다. 결과는 별도 폴더에 저장하며 원본 실험을 덮어쓰지 않습니다.

In [ ]:
N_IMAGES = 40       # 이번 파일은 고정된 첫40장만 실행
RUN_FOLDER = "lvlm_insplad_results/q2_conflict_evidence_dev40_v1"
SUP_ZIP_PATH = ""   # 자동 검색 실패 시에만 실제 Drive 경로 입력
UNSUP_ZIP_PATH = ""

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
SOURCE_SHA256 = 'b55e244c105b0242329d9f648a9f4e881eca63770235231c4bec305c4ffa4c59'
SOURCE_MANIFEST = {'scripts/build_explainable_qa_gold.py': '9acdd42e87c15f9da2ba9bb7c3f89d9e77c42d774a8a8d38f6e783ea544d4f0a', 'scripts/build_gold_v2_dev_set.py': '31c1f17112eea9b279466cdeaa6fdafb43c450be2d6b268136f1b8b8a429f66a', 'scripts/build_q2_conflict_evidence_notebook.py': '0f20409dd6d76e4a9d06f7560660cc9c3a630278b6a8766e23f09e4290f21804', 'scripts/build_retrieved_refs.py': '8f27f2aec2d7eecba622f651c2255a4da98ec435bbf9fd6fe2eedca96ddbed93', 'scripts/dev_part_crop_annotation.py': '39de07b82f31778bfb0a94eac89265dd200296f7fa97e2c60ff6308b180682ea', 'scripts/loc_box_annotation.py': '1d839d6a6b10dda3c4c1d3c7ea16aa17e5d6b072f2df90d7e36c4d64fe316cff', 'scripts/run_dev_prompt_diagnostic.py': '76400860bc603002e6c3aa295b212b02bb2f78fd47b0d52f56c34ec2bd8dc8ab', 'scripts/run_gold_baselines.py': '978cb258c7412a767de2b5f7ee9604e726c29dc41922d656084d751d32869575', 'scripts/run_gold_v2_baselines.py': '72585045062bfc5f5ec5243ca50b34ba7fe4bb871f400bd734712354caec9a2a', 'scripts/run_gold_v2_consistency_metrics.py': '283ab1b1c2c1355895a0a047b9da5cfaef9ed67bf55f0108a0cfd6912e4f2dc2', 'scripts/run_gold_v2_dev_baselines.py': '809eca7a79e59f546107409f2a22f2bd1fda2cef3c77b8ea091b0cd401770fa9', 'scripts/run_gold_v2_q23_candidate.py': '19357ad3507dffd33ff5cd97d7febe36c87ab80a255e09398e380770c8d68499', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'cc8b7937fa75250b99af39ea5fa115cae06cb6f5aa470d0c28f417c4ed246d8a', 'scripts/run_gold_v2_q2_q3_agent_dev.py': '0dec0fd461b115de229913463ed7495552bcfb37a548dea0662f212800d695c8', 'scripts/run_gold_v2_q2_q3_refined_dev.py': '577837b328e7b1a9c8b0ea5a06638a798a9f043d84d58f7e085302b531ba8682', 'scripts/run_gold_v2_reference_ablation.py': 'fff6825808e7b9869c42ec2f54438cb98917c442fcdd2e6797d4554daaed9234', 'scripts/run_gold_v2_verify_agent.py': '0963c6c0c0e19c3bffd7fd7b68dcbd54ade6c51ba64bfd04f41fb0fafe5f966a', 'scripts/run_q2_conflict_evidence_dev.py': '7f830216585b43d6118dfe648549cbcce8c1f78028d3675b3437431f8ab36da6', 'src/insplad_benchmark/__init__.py': '5d9678c34324691fa8d7d5b507501945bb6eafd32e53ef6dedf1979e005cca49', 'src/insplad_benchmark/dataset.py': 'f724c3c5a76f4caa8fa86ac3d3fddef8fed0c466943f979b815f380fbd46652c', 'src/insplad_benchmark/hf_adapters.py': 'dbf178f1a6263c371fc8c7db192a33bdd252910b63267583d5bc24db11f684ab', 'src/insplad_benchmark/metrics.py': 'bb9386501a38031f3d5993744f02a4a1beeb3610bf2a1daa659100511f04cc9d', 'inputs/dataset.json': '370b5bf32012672f7ebecb0952668eaa7f11430dfe8f2eeb24ecdfa22a97d96f'}
PAYLOAD = base64.b64decode('...')
assert hashlib.sha256(PAYLOAD).hexdigest() == SOURCE_SHA256
CODE = Path('/content/q2_conflict_' + SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(PAYLOAD)) as archive:
    assert set(archive.namelist()) == set(SOURCE_MANIFEST)
    for name in archive.namelist():
        target = CODE / name
        assert target.resolve().is_relative_to(CODE.resolve())
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == SOURCE_MANIFEST[name]
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError('런타임 소스가 다릅니다. 새 런타임으로 실행하세요.')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(content)
sys.path[:0] = [str(CODE/'scripts'), str(CODE/'src')]
import run_q2_conflict_evidence_dev as experiment
DATASET = CODE/'inputs/dataset.json'
assert N_IMAGES == 40, '확대 실행은 40장 결과 검토 후 별도 파일로 진행합니다.'
OUT = DRIVE / RUN_FOLDER
assert OUT.name.startswith('q2_conflict_evidence_dev')
print('실제 추론 준비:', N_IMAGES, '장 / 충돌10건 / 신규40호출. 소스:', SOURCE_SHA256)

In [ ]:
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20*1024**3:
    raise RuntimeError('L4/A100 VRAM20GB 이상 런타임을 선택하세요.')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.14.0',
    'accelerate==1.14.0','sentencepiece==0.2.2','numpy>=1.26','pillow>=10','timm'],check=True)
def find_zip(name,size,override):
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:
            raise FileNotFoundError('입력한 ZIP 경로/크기 확인: '+str(path))
        return path
    for path in (DRIVE/name,DRIVE/'lvlm_insplad_data'/name):
        if path.is_file() and path.stat().st_size==size:
            return path
    matches=[p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size]
    if not matches:
        raise FileNotFoundError(name+'를 Drive에서 찾을 수 없습니다.')
    return sorted(matches)[0]
SUP=find_zip('supervised_fault_classification.zip',874753167,SUP_ZIP_PATH)
UNSUP=find_zip('unsupervised_anomaly_detection.zip',1170967719,UNSUP_ZIP_PATH)
from huggingface_hub import snapshot_download
snapshot=snapshot_download(experiment.MODEL_ID,revision=experiment.MODEL_REVISION)
command=[sys.executable,str(CODE/'scripts/run_q2_conflict_evidence_dev.py'),
    '--dataset',str(DATASET),'--sup-zip',str(SUP),'--unsup-zip',str(UNSUP),
    '--model-path',snapshot,'--out-dir',str(OUT),'--n-images',str(N_IMAGES)]
# Runner failures still reach the ZIP export cell, preserving diagnostics.
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (DRIVE/'q2_conflict_evidence_dev40_console.log').open('a',encoding='utf-8') as log:
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait()
print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/'q2_conflict_evidence_dev40_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',PAYLOAD)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    log=DRIVE/'q2_conflict_evidence_dev40_console.log'
    if log.exists():archive.write(log,'console.log')
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
print('ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
status_path=OUT/'run_status.json'
status=json.loads(status_path.read_text()) if status_path.exists() else {'complete':False,'status':'runner failed before status'}
print('RUN STATUS',status)
if RETURN_CODE != 0 or not status.get('complete'):
    print('미완료/무효 정책을 성공으로 해석하지 마세요. ZIP에 원시 오류·응답을 보존했습니다.')
files.download(str(RESULT))